In [80]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import re 
from datetime import datetime 

df = pd.read_csv('Data/train_set_.csv', sep=';')
df_test = pd.read_csv('Data/test_set_without lables_.csv', sep=';') 
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5848 entries, 0 to 5847
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ID                 5848 non-null   int64  
 1   Name               5848 non-null   object 
 2   Location           5848 non-null   object 
 3   Year               5848 non-null   int64  
 4   Kilometers_Driven  5848 non-null   int64  
 5   Fuel_Type          5848 non-null   object 
 6   Transmission       5848 non-null   object 
 7   Owner_Type         5848 non-null   object 
 8   Mileage            5846 non-null   object 
 9   Engine             5813 non-null   object 
 10  Power              5813 non-null   object 
 11  Seats              5807 non-null   float64
 12  Price              5848 non-null   float64
dtypes: float64(2), int64(3), object(8)
memory usage: 594.1+ KB


In [81]:
# dropping useless features 
df = df.drop(['ID', 'Name', 'Location'], axis=1)  
df_test = df_test.drop(['ID', 'Name', 'Location'], axis=1) 
# df.loc[df['Year'] == 2016, 'Mileage'].plot(kind='hist', bins=20, edgecolor='black')
# df['Mileag_Unit'] = df['Mileage'].str.extract(r'^([\d.]+)\s*([A-Za-z/]+)')[1]
# both are electric 

### ---- order ----- Year	Kilometers_Driven	Fuel_Type	Transmission	Owner_Type	Mileage	Engine	Power	Seats	Price

In [82]:
df

,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price
0,2010,72000,CNG,Manual,First,26.6 km/kg,998 CC,58.16 bhp,5.0,1.75
1,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,12.50
2,2011,46000,Petrol,Manual,First,18.2 kmpl,1199 CC,88.7 bhp,5.0,4.50
3,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,6.00
4,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,17.74
...,...,...,...,...,...,...,...,...,...,...
5843,2014,27365,Diesel,Manual,First,28.4 kmpl,1248 CC,74 bhp,5.0,4.75
5844,2015,100000,Diesel,Manual,First,24.4 kmpl,1120 CC,71 bhp,5.0,4.00
5845,2012,55000,Diesel,Manual,Second,14.0 kmpl,2498 CC,112 bhp,8.0,2.90
5846,2013,46000,Petrol,Manual,First,18.9 kmpl,998 CC,67.1 bhp,5.0,2.65


In [83]:
# remove nan values * only to the train data frame 
# drop nan records 
drop_index = df[df['Mileage'].isna()].index
df = df.drop(index=drop_index) 
drop_index = df[df['Engine'].isna()].index 
df = df.drop(index=drop_index) 

# filling missing value seats with the mode 
seats_mode = df['Seats'].mode()[0]
df['Seats'] = df['Seats'].fillna(seats_mode) 

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5811 entries, 0 to 5847
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Year               5811 non-null   int64  
 1   Kilometers_Driven  5811 non-null   int64  
 2   Fuel_Type          5811 non-null   object 
 3   Transmission       5811 non-null   object 
 4   Owner_Type         5811 non-null   object 
 5   Mileage            5811 non-null   object 
 6   Engine             5811 non-null   object 
 7   Power              5811 non-null   object 
 8   Seats              5811 non-null   float64
 9   Price              5811 non-null   float64
dtypes: float64(2), int64(2), object(6)
memory usage: 499.4+ KB


In [84]:
df_test

,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats
0,2017,13372,Petrol,Automatic,First,19.0 kmpl,2996 CC,362.07 bhp,2.0
1,2015,41000,Diesel,Automatic,First,15.3 kmpl,2993 CC,258 bhp,5.0
2,2016,44414,Diesel,Automatic,First,17.9 kmpl,2143 CC,170 bhp,5.0
3,2018,36091,Diesel,Automatic,First,12.7 kmpl,2179 CC,187.7 bhp,5.0
4,2012,56000,Diesel,Automatic,First,11.8 kmpl,2967 CC,246.7 bhp,5.0
...,...,...,...,...,...,...,...,...,...
166,2017,50794,Diesel,Automatic,First,17.9 kmpl,2143 CC,170 bhp,5.0
167,2015,8000,Petrol,Automatic,First,12.5 kmpl,5000 CC,488.1 bhp,2.0
168,2018,29091,Diesel,Automatic,First,13.22 kmpl,2967 CC,241.4 bhp,5.0
169,2016,16000,Diesel,Automatic,First,14.69 kmpl,2993 CC,258 bhp,5.0


In [85]:
# creating the age feature for two data set (training and testing) 
curr_year = datetime.now().year

# the training df 
df['Age'] = curr_year - df['Year'] 
df = df.drop('Year', axis=1) 

# the testing df 
df_test['Age'] = curr_year - df_test['Year'] 
df_test = df_test.drop('Year', axis=1) 

In [86]:
# putting the price feature in the end of data frame (only on the training data) 
df['Price'] = df.pop('Price') 
df

,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Age,Price
0,72000,CNG,Manual,First,26.6 km/kg,998 CC,58.16 bhp,5.0,16,1.75
1,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,11,12.50
2,46000,Petrol,Manual,First,18.2 kmpl,1199 CC,88.7 bhp,5.0,15,4.50
3,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,14,6.00
4,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,13,17.74
...,...,...,...,...,...,...,...,...,...,...
5843,27365,Diesel,Manual,First,28.4 kmpl,1248 CC,74 bhp,5.0,12,4.75
5844,100000,Diesel,Manual,First,24.4 kmpl,1120 CC,71 bhp,5.0,11,4.00
5845,55000,Diesel,Manual,Second,14.0 kmpl,2498 CC,112 bhp,8.0,14,2.90
5846,46000,Petrol,Manual,First,18.9 kmpl,998 CC,67.1 bhp,5.0,13,2.65


In [87]:
# removing the units from the data (training and testing) 

# training 
df['Mileage'] = df['Mileage'].str.extract(r'^([\d.]+)\s*([A-Za-z/]+)')[0].astype(float)
df['Engine'] = df['Engine'].str.extract(r'^([\d.]+)\s*([A-Za-z/]+)')[0].astype(float)
df['Power'] = df['Power'].str.extract(r'^([\d.]+)\s*([A-Za-z/]+)')[0].astype(float)

# testing 
df_test['Mileage'] = df_test['Mileage'].str.extract(r'^([\d.]+)\s*([A-Za-z/]+)')[0].astype(float)
df_test['Engine'] = df_test['Engine'].str.extract(r'^([\d.]+)\s*([A-Za-z/]+)')[0].astype(float)
df_test['Power'] = df_test['Power'].str.extract(r'^([\d.]+)\s*([A-Za-z/]+)')[0].astype(float)


In [88]:
# filing the power feature in the training data set withe the mode (only the training data set) 
power_mode = df['Power'].mode()[0]
df['Power'] = df['Power'].fillna(power_mode)

In [89]:
df.info()
df_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5811 entries, 0 to 5847
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Kilometers_Driven  5811 non-null   int64  
 1   Fuel_Type          5811 non-null   object 
 2   Transmission       5811 non-null   object 
 3   Owner_Type         5811 non-null   object 
 4   Mileage            5811 non-null   float64
 5   Engine             5811 non-null   float64
 6   Power              5811 non-null   float64
 7   Seats              5811 non-null   float64
 8   Age                5811 non-null   int64  
 9   Price              5811 non-null   float64
dtypes: float64(5), int64(2), object(3)
memory usage: 499.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171 entries, 0 to 170
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Kilometers_Driven  171 non-null    int64  
 1   Fuel_

In [90]:
object_col = [col for col in df.columns if df[col].dtype == 'object'] # discrete values -> use one-hot-encoding 

In [91]:
object_col

['Fuel_Type', 'Transmission', 'Owner_Type']

In [92]:
# encoding the discrete values using one hot encoding (this applies on both the training and testing) 
# from sklearn.preprocessing import OneHotEncoder
# encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
# encoder.fit(df[object_col]) 

# training 
df = pd.get_dummies(df, object_col)

# testing 
df_test = pd.get_dummies(df_test, object_col) 

# align test with training columns 
df_test = df_test.reindex(columns=df.columns, fill_value=0)

# pop the price column in the training 
df['Price'] = df.pop('Price') 

# drop the price from the tesing data 
df_test = df_test.drop('Price', axis=1) 

In [93]:
df.head(2)

,Kilometers_Driven,Mileage,Engine,Power,Seats,Age,Fuel_Type_CNG,Fuel_Type_Diesel,Fuel_Type_LPG,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual,Owner_Type_First,Owner_Type_Fourth & Above,Owner_Type_Second,Owner_Type_Third,Price
0,72000,26.60,998.0,58.16,5.0,16,True,False,False,False,False,True,True,False,False,False,1.75
1,41000,19.67,1582.0,126.20,5.0,11,False,True,False,False,False,True,True,False,False,False,12.50


In [94]:
df_test.head(2)

,Kilometers_Driven,Mileage,Engine,Power,Seats,Age,Fuel_Type_CNG,Fuel_Type_Diesel,Fuel_Type_LPG,Fuel_Type_Petrol,Transmission_Automatic,Transmission_Manual,Owner_Type_First,Owner_Type_Fourth & Above,Owner_Type_Second,Owner_Type_Third
0,13372,19.0,2996.0,362.07,2.0,9,0,False,0,True,True,False,True,0,False,False
1,41000,15.3,2993.0,258.00,5.0,11,0,True,0,False,True,False,True,0,False,False


In [96]:
df.info()
df_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5811 entries, 0 to 5847
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Kilometers_Driven          5811 non-null   int64  
 1   Mileage                    5811 non-null   float64
 2   Engine                     5811 non-null   float64
 3   Power                      5811 non-null   float64
 4   Seats                      5811 non-null   float64
 5   Age                        5811 non-null   int64  
 6   Fuel_Type_CNG              5811 non-null   bool   
 7   Fuel_Type_Diesel           5811 non-null   bool   
 8   Fuel_Type_LPG              5811 non-null   bool   
 9   Fuel_Type_Petrol           5811 non-null   bool   
 10  Transmission_Automatic     5811 non-null   bool   
 11  Transmission_Manual        5811 non-null   bool   
 12  Owner_Type_First           5811 non-null   bool   
 13  Owner_Type_Fourth & Above  5811 non-null   bool   
 1

In [98]:
# for some reason we lost some values in the testing set we don't know why but for confiening 
df_test = df_test.fillna(df_test.mean()) 

In [99]:
df.info()
df_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5811 entries, 0 to 5847
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Kilometers_Driven          5811 non-null   int64  
 1   Mileage                    5811 non-null   float64
 2   Engine                     5811 non-null   float64
 3   Power                      5811 non-null   float64
 4   Seats                      5811 non-null   float64
 5   Age                        5811 non-null   int64  
 6   Fuel_Type_CNG              5811 non-null   bool   
 7   Fuel_Type_Diesel           5811 non-null   bool   
 8   Fuel_Type_LPG              5811 non-null   bool   
 9   Fuel_Type_Petrol           5811 non-null   bool   
 10  Transmission_Automatic     5811 non-null   bool   
 11  Transmission_Manual        5811 non-null   bool   
 12  Owner_Type_First           5811 non-null   bool   
 13  Owner_Type_Fourth & Above  5811 non-null   bool   
 1

In [101]:
df.to_csv('Data2/clean_train_data.csv', index=False)
df_test.to_csv('Data2/clean_test_data.csv', index=False) 